# 06 Regime Overlay Validation Diagnostic

Purpose: score and walk-forward validate optional regime overlay candidates created by Notebook 05.

Scope note: This notebook validates 05 overlays. If overlays fail WFV, they are parked and not promoted downstream.

Scope boundaries:
- This notebook validates regime overlay candidates only.
- It does not change base alpha formulas, scoring thresholds, WFV thresholds, or downstream notebooks.
- It writes only separate `regime_context_alpha_*` validation tables plus a diagnostic decision table.
- Overlay candidates require explicit future promotion before stress testing, survivor freeze, or portfolio construction.


## 1. Imports and Config

In [1]:
from pathlib import Path
import gc
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.alpha_scoring import (
    APPROVED_FOR_ALPHA_WFV,
    WATCHLIST_ALPHA_WFV,
    build_alpha_best_horizon_summary,
    build_alpha_scoring_gate,
)
from src.alpha_wfv import (
    apply_alpha_wfv_gate,
    build_alpha_wfv_failure_breakdown,
    build_alpha_wfv_winner_summary,
    run_wfv_for_alpha_candidates,
    summarize_alpha_wfv_results,
)
from src.db import load_ohlcv_panels
from src.regime_context_alpha_validation import (
    load_regime_context_alpha_validation_inputs,
    prepare_regime_context_alpha_scoring_input,
    score_regime_context_alpha_library,
    select_approved_regime_context_alphas,
)
from src.regime_context_alpha_validation_storage import (
    REGIME_CONTEXT_ALPHA_VALIDATION_TABLES,
    REGIME_OVERLAY_DIAGNOSTIC_DECISION_TABLES,
    save_regime_context_alpha_validation_outputs,
    save_regime_overlay_diagnostic_decision,
)
from src.run_config import get_sqlite_db_path, make_run_id, make_run_timestamp
from src.walkforward import generate_walkforward_windows

DB_PATH = get_sqlite_db_path()
REGIME_CONTEXT_ALPHA_SCORING_VERSION = "phase6_regime_context_alpha_scoring_v2"
REGIME_CONTEXT_ALPHA_WFV_VERSION = "phase6_regime_context_alpha_wfv_v2"
REGIME_OVERLAY_DIAGNOSTIC_VERSION = "phase6_regime_overlay_diagnostic_v1"
IC_METHOD = "spearman"
HORIZONS = [1, 5, 10, 20]

V3_DYNAMIC_BASE_ALPHAS = [
    "alpha_hybrid_adaptive_v3",
    "alpha_rolling_ic_dynamic_v3",
    "alpha_regime_blend_dynamic_v3",
    "alpha_decay_aware_dynamic_v3",
]

TRAIN_SIZE = 378
TEST_SIZE = 63
PURGE_SIZE = 20
EMBARGO_SIZE = 5

pd.set_option("display.max_columns", 200)
DB_PATH


PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase6_regime_context_alpha")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase6_regime_context_alpha_20260504_185145', '2026-05-04 18:51:45')

## 3. Load Notebook 05 regime-context alpha artifacts

In [3]:
inputs = load_regime_context_alpha_validation_inputs(db_path=DB_PATH)

input_shapes = pd.DataFrame(
    [
        {"input_name": name, "n_rows": len(df), "n_columns": len(df.columns)}
        for name, df in inputs.items()
    ]
)
display(input_shapes)

,input_name,n_rows,n_columns
0,alpha_long,7591968,6
1,quality,36,14
2,metadata,36,12
3,diagnostics,36,18
4,activation,36,12


## 4. Keep approved regime-context alpha candidates

In [4]:
approved_regime_context_quality = select_approved_regime_context_alphas(
    inputs["quality"],
    metadata=inputs["metadata"],
)
approved_regime_context_names = approved_regime_context_quality["alpha_name"].dropna().astype(str).tolist()

if not approved_regime_context_names:
    raise ValueError("No regime-context alpha candidates are approved for WFV validation.")

OVERLAY_METADATA_COLUMNS = [
    "alpha_name",
    "overlay_type",
    "base_alpha_name",
    "source_alpha_wfv_status",
    "source_alpha_wfv_horizon",
    "source_effective_mean_test_ic",
    "source_persistence_ratio",
]
approved_overlay_metadata = approved_regime_context_quality[
    [column for column in OVERLAY_METADATA_COLUMNS if column in approved_regime_context_quality.columns]
].drop_duplicates("alpha_name")


def attach_overlay_metadata(df: pd.DataFrame) -> pd.DataFrame:
    if "alpha_name" not in df.columns:
        return df
    overlay_columns = [column for column in OVERLAY_METADATA_COLUMNS if column != "alpha_name"]
    cleaned = df.drop(columns=[column for column in overlay_columns if column in df.columns], errors="ignore")
    return cleaned.merge(approved_overlay_metadata, on="alpha_name", how="left")


regime_context_alpha_scoring_input = prepare_regime_context_alpha_scoring_input(
    inputs["alpha_long"],
    approved_regime_context_quality,
)
found_alpha_names = sorted(regime_context_alpha_scoring_input["alpha_name"].dropna().astype(str).unique().tolist())
missing_alpha_names = sorted(set(approved_regime_context_names).difference(found_alpha_names))

approved_overlay_count_by_overlay_type = (
    approved_regime_context_quality["overlay_type"]
    .value_counts(dropna=False)
    .rename_axis("overlay_type")
    .reset_index(name="n_approved_overlays")
    .sort_values("overlay_type")
    if "overlay_type" in approved_regime_context_quality.columns
    else pd.DataFrame(columns=["overlay_type", "n_approved_overlays"])
)
approved_overlay_count_by_base_alpha_name = (
    approved_regime_context_quality["base_alpha_name"]
    .value_counts(dropna=False)
    .rename_axis("base_alpha_name")
    .reset_index(name="n_approved_overlays")
    .sort_values("base_alpha_name")
    if "base_alpha_name" in approved_regime_context_quality.columns
    else pd.DataFrame(columns=["base_alpha_name", "n_approved_overlays"])
)
v3_overlay_inclusion_check = pd.DataFrame(
    [
        {
            "base_alpha_name": base_alpha_name,
            "n_approved_overlays": int(
                approved_regime_context_quality.get("base_alpha_name", pd.Series(dtype=object))
                .astype(str)
                .eq(base_alpha_name)
                .sum()
            ),
            "included_flag": bool(
                approved_regime_context_quality.get("base_alpha_name", pd.Series(dtype=object))
                .astype(str)
                .eq(base_alpha_name)
                .any()
            ),
        }
        for base_alpha_name in V3_DYNAMIC_BASE_ALPHAS
    ]
)

print(f"Approved regime-context overlay candidates: {len(approved_regime_context_names)}")
print(f"Alpha observations for scoring: {len(regime_context_alpha_scoring_input):,}")
print(f"Horizons: {HORIZONS}")
print(f"Missing approved overlays from long table: {missing_alpha_names}")

display(approved_overlay_count_by_overlay_type)
display(approved_overlay_count_by_base_alpha_name)
display(v3_overlay_inclusion_check)
display(approved_regime_context_quality.sort_values("alpha_name"))


Approved regime-context overlay candidates: 32
Alpha observations for scoring: 6,748,416
Horizons: [1, 5, 10, 20]
Missing approved overlays from long table: []


,overlay_type,n_approved_overlays
0,base_passthrough,8
1,defensive_downscale,8
2,mild_regime_scaled,8
3,volatility_stress_scaled,8


,base_alpha_name,n_approved_overlays
0,alpha_decay_aware_dynamic_v3,4
1,alpha_equal_weight_research_v1,4
2,alpha_health_weighted_research_v1,4
3,alpha_hybrid_adaptive_v3,4
4,alpha_persistence_blend_v2,4
5,alpha_regime_blend_dynamic_v3,4
6,alpha_rolling_ic_dynamic_v3,4
7,alpha_smooth_regime_weighted_v2,4


,base_alpha_name,n_approved_overlays,included_flag
0,alpha_hybrid_adaptive_v3,4,True
1,alpha_rolling_ic_dynamic_v3,4,True
2,alpha_regime_blend_dynamic_v3,4,True
3,alpha_decay_aware_dynamic_v3,4,True


,alpha_name,finite_pct,missing_pct,max_abs_alpha,avg_turnover_proxy,turnover_risk_flag,n_dates,n_tickers,first_valid_date,last_valid_date,status,quality_notes,run_id,regime_context_version,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
0,alpha_decay_aware_dynamic_v3__base_passthrough,0.980587,0.019413,3.000000,1.582108,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
1,alpha_decay_aware_dynamic_v3__defensive_downscale,0.980587,0.019413,3.000000,1.582108,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
2,alpha_decay_aware_dynamic_v3__mild_regime_scaled,0.980587,0.019413,3.000000,1.582108,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
3,alpha_decay_aware_dynamic_v3__volatility_stres...,0.980587,0.019413,3.000000,1.582108,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,alpha_equal_weight_research_v1__base_passthrough,0.980587,0.019413,2.292711,1.658849,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_equal_weight_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
5,alpha_equal_weight_research_v1__defensive_down...,0.980587,0.019413,2.292711,1.658797,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_equal_weight_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
6,alpha_equal_weight_research_v1__mild_regime_sc...,0.980587,0.019413,2.292711,1.658861,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_equal_weight_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
7,alpha_equal_weight_research_v1__volatility_str...,0.980587,0.019413,2.292711,1.658870,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_equal_weight_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
8,alpha_health_weighted_research_v1__base_passth...,0.980587,0.019413,2.268331,1.664005,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_health_weighted_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50
9,alpha_health_weighted_research_v1__defensive_d...,0.980587,0.019413,2.268331,1.664007,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04

## 5. Display activation diagnostics

In [5]:
activation_diagnostics = inputs["activation"].copy()
display(activation_diagnostics.sort_values("alpha_name"))

diagnostics = inputs["diagnostics"].copy()
display(diagnostics.sort_values("alpha_name"))

,alpha_name,overlay_type,base_alpha_name,scaling_rule,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,n_scaled_dates,n_total_dates,run_id,regime_context_version
12,alpha_decay_aware_dynamic_v3__base_passthrough,base_passthrough,alpha_decay_aware_dynamic_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
14,alpha_decay_aware_dynamic_v3__defensive_downscale,defensive_downscale,alpha_decay_aware_dynamic_v3,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
13,alpha_decay_aware_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_decay_aware_dynamic_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
15,alpha_decay_aware_dynamic_v3__volatility_stres...,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
32,alpha_diversified_research_v2__base_passthrough,base_passthrough,alpha_diversified_research_v2,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
34,alpha_diversified_research_v2__defensive_downs...,defensive_downscale,alpha_diversified_research_v2,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
33,alpha_diversified_research_v2__mild_regime_scaled,mild_regime_scaled,alpha_diversified_research_v2,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
35,alpha_diversified_research_v2__volatility_stre...,volatility_stress_scaled,alpha_diversified_research_v2,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
24,alpha_equal_weight_research_v1__base_passthrough,base_passthrough,alpha_equal_weight_research_v1,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
26,alpha_equal_weight_research_v1__defensive_down...,defensive_downscale,alpha_equal_weight_research_v1,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


,alpha_name,finite_pct,missing_pct,n_dates,n_tickers,mean_abs_alpha,alpha_std,max_abs_alpha,avg_turnover_proxy,median_turnover_proxy,max_turnover_proxy,turnover_risk_flag,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,run_id,regime_context_version
12,alpha_decay_aware_dynamic_v3__base_passthrough,0.980587,0.019413,2088,101,0.777010,0.992249,3.000000,1.582108,1.445545,5.783505,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
14,alpha_decay_aware_dynamic_v3__defensive_downscale,0.980587,0.019413,2088,101,0.777010,0.992249,3.000000,1.582108,1.445545,5.783505,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
13,alpha_decay_aware_dynamic_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.777010,0.992249,3.000000,1.582108,1.445545,5.783505,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
15,alpha_decay_aware_dynamic_v3__volatility_stres...,0.980587,0.019413,2088,101,0.777010,0.992249,3.000000,1.582108,1.445545,5.783505,LOW_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
32,alpha_diversified_research_v2__base_passthrough,0.988729,0.011271,2088,101,0.826259,0.994989,3.000000,3.022672,2.831683,11.824742,HIGH_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
34,alpha_diversified_research_v2__defensive_downs...,0.988729,0.011271,2088,101,0.826259,0.994989,3.000000,3.022672,2.831683,11.824742,HIGH_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
33,alpha_diversified_research_v2__mild_regime_scaled,0.988729,0.011271,2088,101,0.826259,0.994989,3.000000,3.022672,2.831683,11.824742,HIGH_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
35,alpha_diversified_research_v2__volatility_stre...,0.988729,0.011271,2088,101,0.826259,0.994989,3.000000,3.022672,2.831683,11.824742,HIGH_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
24,alpha_equal_weight_research_v1__base_passthrough,0.980587,0.019413,2088,101,0.840670,0.994990,2.292711,1.658849,1.494949,6.840206,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
26,alpha_equal_weight_research_v1__defensive_down...,0.980587,0.019413,2088,101,0.840670,0.994990,2.292711,1.658797,1.495050,6.840206,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


## 6. Load clean close prices

In [6]:
ohlcv = load_ohlcv_panels(current=True, db_path=DB_PATH)
close_prices = ohlcv["close"]

close_prices.shape

(2088, 101)

## 7. Score regime-context alpha candidates

In [7]:
regime_context_alpha_scores = score_regime_context_alpha_library(
    alpha_long_df=regime_context_alpha_scoring_input,
    close_prices=close_prices,
    horizons=HORIZONS,
    method=IC_METHOD,
)
regime_context_alpha_scores = attach_overlay_metadata(regime_context_alpha_scores)

score_display = regime_context_alpha_scores.sort_values(
    "mean_ic",
    key=lambda series: series.abs(),
    ascending=False,
)
display(score_display)


,alpha_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,alpha_version,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
103,alpha_rolling_ic_dynamic_v3__defensive_downscale,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
107,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
99,alpha_rolling_ic_dynamic_v3__base_passthrough,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
111,alpha_rolling_ic_dynamic_v3__volatility_stress...,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
59,alpha_hybrid_adaptive_v3__mild_regime_scaled,20,spearman,204774,0.051068,0.052888,0.226222,0.225742,0.488735,0.578125,0.019601,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,alpha_regime_blend_dynamic_v3__defensive_downs...,1,spearman,206693,0.006459,0.005416,0.260223,0.024820,0.491657,0.508466,0.010413,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75
12,alpha_decay_aware_dynamic_v3__volatility_stres...,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
8,alpha_decay_aware_dynamic_v3__mild_regime_scaled,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,alpha_decay_aware_dynamic_v3__defensive_downscale,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75


## 8. Build scoring gate and best horizon

In [8]:
regime_context_alpha_best_horizon = attach_overlay_metadata(
    build_alpha_best_horizon_summary(regime_context_alpha_scores)
)
regime_context_alpha_scoring_gate = attach_overlay_metadata(
    build_alpha_scoring_gate(regime_context_alpha_scores)
)

scoring_gate_counts = (
    regime_context_alpha_scoring_gate["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_alpha_horizons")
)

display(scoring_gate_counts)
display(regime_context_alpha_best_horizon)
display(regime_context_alpha_scoring_gate.sort_values("abs_mean_ic", ascending=False))


,status,n_alpha_horizons
0,APPROVED_FOR_ALPHA_WFV,80
1,REJECTED_ALPHA_LOW_SIGNAL,28
2,WATCHLIST_ALPHA_WFV,20


,alpha_name,best_horizon,best_mean_ic,best_abs_mean_ic,best_ic_ir,best_positive_ic_rate,best_hit_rate,alpha_direction,alpha_strength,alpha_version,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
0,alpha_rolling_ic_dynamic_v3__volatility_stress...,20,0.054179,0.054179,0.266674,0.608398,0.489605,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
1,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,20,0.054179,0.054179,0.266674,0.608398,0.489605,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
2,alpha_rolling_ic_dynamic_v3__defensive_downscale,20,0.054179,0.054179,0.266674,0.608398,0.489605,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
3,alpha_rolling_ic_dynamic_v3__base_passthrough,20,0.054179,0.054179,0.266674,0.608398,0.489605,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
4,alpha_hybrid_adaptive_v3__defensive_downscale,20,0.051068,0.051068,0.225742,0.578125,0.488735,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,defensive_downscale,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
5,alpha_hybrid_adaptive_v3__volatility_stress_sc...,20,0.051068,0.051068,0.225742,0.578125,0.488735,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
6,alpha_hybrid_adaptive_v3__mild_regime_scaled,20,0.051068,0.051068,0.225742,0.578125,0.488735,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
7,alpha_hybrid_adaptive_v3__base_passthrough,20,0.051068,0.051068,0.225742,0.578125,0.488735,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,base_passthrough,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
8,alpha_regime_blend_dynamic_v3__mild_regime_scaled,20,0.047520,0.047520,0.203088,0.575195,0.489141,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,mild_regime_scaled,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75
9,alpha_regime_blend_dynamic_v3__volatility_stre...,20,0.047520,0.047520,0.203088,0.575195,0.489141,POSITIVE_EDGE,STRONG,phase5_regime_overlay_alpha_v3,volatility_stress_scaled,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75


,alpha_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,alpha_version,abs_mean_ic,abs_ic_ir,alpha_direction,alpha_strength,status,scoring_gate_notes,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
103,alpha_rolling_ic_dynamic_v3__defensive_downscale,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",defensive_downscale,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
107,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
99,alpha_rolling_ic_dynamic_v3__base_passthrough,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
111,alpha_rolling_ic_dynamic_v3__volatility_stress...,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
59,alpha_hybrid_adaptive_v3__mild_regime_scaled,20,spearman,204774,0.051068,0.052888,0.226222,0.225742,0.488735,0.578125,0.019601,phase5_regime_overlay_alpha_v3,0.051068,0.225742,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",mild_regime_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,alpha_regime_blend_dynamic_v3__defensive_downs...,1,spearman,206693,0.006459,0.005416,0.260223,0.024820,0.491657,0.508466,0.010413,phase5_regime_overlay_alpha_v3,0.006459,0.024820,POSITIVE_EDGE,NO_SIGNAL,REJECTED_ALPHA_LOW_SIGNAL,Fails preliminary alpha predictive scoring thr...,defensive_downscale,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75
12,alpha_decay_aware_dynamic_v3__volatility_stres...,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,0.006343,0.024402,POSITIVE_EDGE,NO_SIGNAL,REJECTED_ALPHA_LOW_SIGNAL,Fails preliminary alpha predictive scoring thr...,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
8,alpha_decay_aware_dynamic_v3__mild_regime_scaled,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,0.006343,0.024402,POSITIVE_EDGE,NO_SIGNAL,REJECTED_ALPHA_LOW_SIGNAL,Fails preliminary alpha predictive scoring thr...,mild_regime_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,alpha_decay_aware_dynamic_v3__defensive_downscale,1,spearman,206693,0.006343,0.005168,0.259956,0.024402,0.491720,0.507015,0.010413,phase5_regime_overlay_alpha_v3,0.006343,0.024402,POSITIVE_EDGE,NO_SIGNAL,REJECTED_ALPHA_LOW_SIGNAL,Fails preliminary alpha predictive scoring thr...,defensive_downscale,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75


## 9. Select regime-context alpha WFV candidates

In [9]:
regime_context_alpha_wfv_candidates = regime_context_alpha_scoring_gate.loc[
    regime_context_alpha_scoring_gate["status"].isin([APPROVED_FOR_ALPHA_WFV, WATCHLIST_ALPHA_WFV])
].copy()
regime_context_alpha_wfv_candidates["candidate_tier"] = regime_context_alpha_wfv_candidates["status"].map(
    {
        APPROVED_FOR_ALPHA_WFV: "PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE",
        WATCHLIST_ALPHA_WFV: "WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE",
    }
)
regime_context_alpha_wfv_candidates["source_status"] = regime_context_alpha_wfv_candidates["status"]

print(f"Regime-context alpha WFV candidate rows: {len(regime_context_alpha_wfv_candidates)}")
display(regime_context_alpha_wfv_candidates.sort_values("abs_mean_ic", ascending=False))

Regime-context alpha WFV candidate rows: 100


,alpha_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,alpha_version,abs_mean_ic,abs_ic_ir,alpha_direction,alpha_strength,status,scoring_gate_notes,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio,candidate_tier,source_status
111,alpha_rolling_ic_dynamic_v3__volatility_stress...,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,APPROVED_FOR_ALPHA_WFV
107,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,APPROVED_FOR_ALPHA_WFV
103,alpha_rolling_ic_dynamic_v3__defensive_downscale,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",defensive_downscale,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,APPROVED_FOR_ALPHA_WFV
99,alpha_rolling_ic_dynamic_v3__base_passthrough,20,spearman,204774,0.054179,0.056876,0.203164,0.266674,0.489605,0.608398,0.019601,phase5_regime_overlay_alpha_v3,0.054179,0.266674,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,APPROVED_FOR_ALPHA_WFV
55,alpha_hybrid_adaptive_v3__defensive_downscale,20,spearman,204774,0.051068,0.052888,0.226222,0.225742,0.488735,0.578125,0.019601,phase5_regime_overlay_alpha_v3,0.051068,0.225742,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,"Meets alpha IC, IC IR, and observation thresho...",defensive_downscale,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,APPROVED_FOR_ALPHA_WFV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37,alpha_health_weighted_research_v1__defensive_d...,5,spearman,206289,0.025626,0.030670,0.268276,0.095520,0.497149,0.547261,0.012348,phase5_regime_overlay_alpha_v3,0.025626,0.095520,POSITIVE_EDGE,MODERATE,WATCHLIST_ALPHA_WFV,Meets alpha watchlist IC and observation thres...,defensive_downscale,alpha_health_weighted_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,WATCHLIST_ALPHA_WFV
108,alpha_rolling_ic_dynamic_v3__volatility_stress...,1,spearman,206693,0.013395,0.014077,0.222302,0.060258,0.493627,0.528786,0.010413,phase5_regime_overlay_alpha_v3,0.013395,0.060258,POSITIVE_EDGE,WEAK,WATCHLIST_ALPHA_WFV,Meets alpha watchlist IC and observation thres...,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,WATCHLIST_ALPHA_WFV
104,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,1,spearman,206693,0.013395,0.014077,0.222302,0.060258,0.493627,0.528786,0.010413,phase5_regime_overlay_alpha_v3,0.013395,0.060258,POSITIVE_EDGE,WEAK,WATCHLIST_ALPHA_WFV,Meets alpha watchlist IC and observation thres...,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,WATCHLIST_ALPHA_WFV
100,alpha_rolling_ic_

## 10. Generate WFV windows

In [10]:
regime_context_alpha_wfv_windows = generate_walkforward_windows(
    close_prices.index,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    purge_size=PURGE_SIZE,
    embargo_size=EMBARGO_SIZE,
)

if regime_context_alpha_wfv_windows.empty:
    raise ValueError("WFV configuration produced no windows.")

print(f"Generated WFV windows: {len(regime_context_alpha_wfv_windows)}")
display(regime_context_alpha_wfv_windows)

Generated WFV windows: 4


,window_id,train_start,train_end,test_start,test_end,purge_size,embargo_size,embargo_start,embargo_end,n_train_dates,n_test_dates
0,1,2018-01-02,2019-07-03,2019-08-02,2019-10-30,20,5,2019-10-31,2019-11-06,378,63
1,2,2019-11-07,2021-05-10,2021-06-09,2021-09-07,20,5,2021-09-08,2021-09-14,378,63
2,3,2021-09-15,2023-03-16,2023-04-17,2023-07-17,20,5,2023-07-18,2023-07-24,378,63
3,4,2023-07-25,2025-01-24,2025-02-25,2025-05-23,20,5,2025-05-27,2025-06-02,378,63


## 11. Run regime-context alpha WFV

In [11]:
regime_context_alpha_wfv_window_results = run_wfv_for_alpha_candidates(
    alpha_candidates=regime_context_alpha_wfv_candidates,
    close_prices=close_prices,
    windows=regime_context_alpha_wfv_windows,
    method=IC_METHOD,
    alpha_long_df=regime_context_alpha_scoring_input,
)
regime_context_alpha_wfv_window_results = attach_overlay_metadata(regime_context_alpha_wfv_window_results)

display(regime_context_alpha_wfv_window_results)


,window_id,alpha_name,horizon,candidate_tier,alpha_direction,alpha_strength,source_status,method,train_start,train_end,test_start,test_end,train_mean_ic,test_mean_ic,effective_train_ic,effective_test_ic,train_positive_ic_rate,test_positive_ic_rate,train_n_obs,test_n_obs,direction_flip_warning,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
0,1,alpha_decay_aware_dynamic_v3__base_passthrough,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,APPROVED_FOR_ALPHA_WFV,spearman,2018-01-02,2019-07-03,2019-08-02,2019-10-30,0.013592,0.026046,0.013592,0.026046,0.527933,0.523810,34744,6237,False,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
1,2,alpha_decay_aware_dynamic_v3__base_passthrough,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,APPROVED_FOR_ALPHA_WFV,spearman,2019-11-07,2021-05-10,2021-06-09,2021-09-07,0.014791,-0.035489,0.014791,-0.035489,0.507937,0.523810,37698,6363,True,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
2,3,alpha_decay_aware_dynamic_v3__base_passthrough,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,APPROVED_FOR_ALPHA_WFV,spearman,2021-09-15,2023-03-16,2023-04-17,2023-07-17,0.001810,0.101020,0.001810,0.101020,0.505291,0.682540,38178,6363,False,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
3,4,alpha_decay_aware_dynamic_v3__base_passthrough,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,APPROVED_FOR_ALPHA_WFV,spearman,2023-07-25,2025-01-24,2025-02-25,2025-05-23,0.023412,0.089720,0.023412,0.089720,0.531746,0.619048,38178,6363,False,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,1,alpha_decay_aware_dynamic_v3__base_passthrough,10,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,spearman,2018-01-02,2019-07-03,2019-08-02,2019-10-30,0.011966,0.055909,0.011966,0.055909,0.516760,0.587302,34744,6237,False,base_passthrough,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,4,alpha_smooth_regime_weighted_v2__volatility_st...,10,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,spearman,2023-07-25,2025-01-24,2025-02-25,2025-05-23,0.042536,0.173988,0.042536,0.173988,0.576720,0.666667,38178,6363,False,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
396,1,alpha_smooth_regime_weighted_v2__volatility_st...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,spearman,2018-01-02,2019-07-03,2019-08-02,2019-10-30,0.003821,0.125476,0.003821,0.125476,0.474860,0.714286,34744,6237,False,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
397,2,alpha_smooth_regime_weighted_v2__volatility_st...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,spearman,2019-11-07,2021-05-10,2021-06-09,2021-09-07,0.043342,-0.025757,0.043342,-0.025757,0.563492,0.444444,37698,6363,True,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
398,3,alpha_smooth_regime_weighted_v2__volatility_st...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,APPROVED_FOR_ALPHA_WFV,spearman,2021-09-15,2023-03-16,2023-04-17,2023-07-17,-0.023405,0.236462,-0.023405,0.236462,0.441799,0.952381,38178,6363,False,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50


## 12. Build WFV summary and gate

In [12]:
regime_context_alpha_wfv_summary = attach_overlay_metadata(
    summarize_alpha_wfv_results(regime_context_alpha_wfv_window_results)
)
regime_context_alpha_wfv_gate = attach_overlay_metadata(
    apply_alpha_wfv_gate(regime_context_alpha_wfv_summary)
)
regime_context_alpha_wfv_failure_breakdown = build_alpha_wfv_failure_breakdown(regime_context_alpha_wfv_gate)
regime_context_alpha_wfv_winner_summary = attach_overlay_metadata(
    build_alpha_wfv_winner_summary(regime_context_alpha_wfv_gate)
)

regime_context_alpha_wfv_gate_counts = (
    regime_context_alpha_wfv_gate["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_alpha_horizons")
    if "status" in regime_context_alpha_wfv_gate.columns
    else pd.DataFrame(columns=["status", "n_alpha_horizons"])
)

display(regime_context_alpha_wfv_gate_counts)
display(regime_context_alpha_wfv_winner_summary)
display(regime_context_alpha_wfv_failure_breakdown)
display(
    regime_context_alpha_wfv_gate.sort_values("effective_mean_test_ic", ascending=False)
    if not regime_context_alpha_wfv_gate.empty
    else regime_context_alpha_wfv_gate
)


,status,n_alpha_horizons
0,REJECTED_ALPHA_WFV,100


,alpha_name,horizon,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,candidate_tier,alpha_direction,alpha_strength,status,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio


,failure_reason,count,pct_of_candidates
0,direction flip,100,1.0


,alpha_name,horizon,candidate_tier,alpha_direction,alpha_strength,n_windows,mean_train_ic,mean_test_ic,effective_mean_train_ic,effective_mean_test_ic,median_test_ic,effective_median_test_ic,test_ic_std,effective_test_ic_std,test_ic_ir,effective_test_ic_ir,test_positive_ic_rate,persistence_ratio,sign_consistency,direction_flip_warning,degradation_ratio,n_positive_test_windows,n_negative_test_windows,abs_mean_test_ic,abs_test_ic_ir,status,wfv_gate_notes,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
99,alpha_smooth_regime_weighted_v2__volatility_st...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740579,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
96,alpha_smooth_regime_weighted_v2__mild_regime_s...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740554,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,mild_regime_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
90,alpha_smooth_regime_weighted_v2__base_passthrough,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740604,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,base_passthrough,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
93,alpha_smooth_regime_weighted_v2__defensive_dow...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180968,0.180968,0.128659,0.128659,1.145877,1.145877,0.75,0.50,0.75,True,6.740596,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,defensive_downscale,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
35,alpha_health_weighted_research_v1__volatility_...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021728,0.146375,0.021728,0.146375,0.178983,0.178983,0.127684,0.127684,1.146382,1.146382,0.75,0.50,0.75,True,6.736750,3,1,0.146375,1.146382,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_health_weighted_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,alpha_decay_aware_dynamic_v3__volatility_stres...,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,4,0.013401,0.045324,0.013401,0.045324,0.057883,0.057883,0.063181,0.063181,0.717371,0.717371,0.75,0.75,0.75,True,3.382092,3,1,0.045324,0.717371,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
72,alpha_rolling_ic_dynamic_v3__base_passthrough,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,WEAK,4,0.006835,0.016490,0.006835,0.016490,0.026330,0.026330,0.029602,0.029602,0.557066,0.557066,0.75,1.00,0.75,True,2.412614,3,1,0.016490,0.557066,REJECTED_ALPHA_WFV,direction flip,base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
84,alpha_rolling_ic_dynamic_v3__volatility_stress...,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,WEAK,4,0.006835,0.016490,0.006835,0.016490,0.026330,0.026330,0.029602,0.029602,0.557066,0.557066,0.75,1.00,0.75,True,2.412614,3,1,0.016490,0.557066,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
80,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE

## 13. Save diagnostic validation outputs to SQLite


In [13]:
saved_paths = save_regime_context_alpha_validation_outputs(
    scores=regime_context_alpha_scores,
    scoring_gate=regime_context_alpha_scoring_gate,
    best_horizon=regime_context_alpha_best_horizon,
    wfv_windows=regime_context_alpha_wfv_windows,
    wfv_window_results=regime_context_alpha_wfv_window_results,
    wfv_summary=regime_context_alpha_wfv_summary,
    wfv_gate=regime_context_alpha_wfv_gate,
    wfv_failure_breakdown=regime_context_alpha_wfv_failure_breakdown,
    wfv_winner_summary=regime_context_alpha_wfv_winner_summary,
    db_path=DB_PATH,
    run_id=run_id,
    scoring_version=REGIME_CONTEXT_ALPHA_SCORING_VERSION,
    wfv_version=REGIME_CONTEXT_ALPHA_WFV_VERSION,
)

n_wfv_approved = int(regime_context_alpha_wfv_gate["status"].eq("APPROVED_ALPHA_WFV").sum()) if "status" in regime_context_alpha_wfv_gate.columns else 0
n_wfv_watchlist = int(regime_context_alpha_wfv_gate["status"].eq("WATCHLIST_ALPHA_WFV").sum()) if "status" in regime_context_alpha_wfv_gate.columns else 0
n_wfv_rejected = int(regime_context_alpha_wfv_gate["status"].eq("REJECTED_ALPHA_WFV").sum()) if "status" in regime_context_alpha_wfv_gate.columns else 0
diagnostic_decision = "REVIEW_OVERLAYS" if n_wfv_approved > 0 else "PARK_OVERLAYS"
decision_notes = (
    "Overlay candidates may be considered for future stress testing, but require explicit promotion."
    if n_wfv_approved > 0
    else "Regime overlays are diagnostic-only and should not feed Notebook 07/08/09."
)

regime_overlay_diagnostic_decision = pd.DataFrame(
    [
        {
            "run_id": run_id,
            "diagnostic_version": REGIME_OVERLAY_DIAGNOSTIC_VERSION,
            "n_overlay_candidates": int(inputs["metadata"]["alpha_name"].nunique()),
            "n_quality_approved": int(len(approved_regime_context_names)),
            "n_wfv_approved": n_wfv_approved,
            "n_wfv_watchlist": n_wfv_watchlist,
            "n_wfv_rejected": n_wfv_rejected,
            "diagnostic_decision": diagnostic_decision,
            "decision_notes": decision_notes,
        }
    ]
)

decision_paths = save_regime_overlay_diagnostic_decision(
    regime_overlay_diagnostic_decision,
    db_path=DB_PATH,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in REGIME_CONTEXT_ALPHA_VALIDATION_TABLES.items()
    ]
    + [
        {
            "artifact": "diagnostic_decision",
            "current_table": REGIME_OVERLAY_DIAGNOSTIC_DECISION_TABLES[0],
            "history_table": REGIME_OVERLAY_DIAGNOSTIC_DECISION_TABLES[1],
            "sqlite_path": str(decision_paths["diagnostic_decision"]),
        }
    ]
)

display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving regime context alpha validation outputs')
display(regime_overlay_diagnostic_decision)


,artifact,current_table,history_table,sqlite_path
0,scores,regime_context_alpha_scores_current,regime_context_alpha_scores_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,scoring_gate,regime_context_alpha_scoring_gate_current,regime_context_alpha_scoring_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,best_horizon,regime_context_alpha_best_horizon_current,regime_context_alpha_best_horizon_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,wfv_windows,regime_context_alpha_wfv_windows_current,regime_context_alpha_wfv_windows_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,wfv_window_results,regime_context_alpha_wfv_window_results_current,regime_context_alpha_wfv_window_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,wfv_summary,regime_context_alpha_wfv_summary_current,regime_context_alpha_wfv_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,wfv_gate,regime_context_alpha_wfv_gate_current,regime_context_alpha_wfv_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
7,wfv_failure_breakdown,regime_context_alpha_wfv_failure_breakdown_cur...,regime_context_alpha_wfv_failure_breakdown_his...,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
8,wfv_winner_summary,regime_context_alpha_wfv_winner_summary_current,regime_context_alpha_wfv_winner_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
9,diagnostic_decision,regime_overlay_diagnostic_decision_current,regime_overlay_diagnostic_decision_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


,run_id,diagnostic_version,n_overlay_candidates,n_quality_approved,n_wfv_approved,n_wfv_watchlist,n_wfv_rejected,diagnostic_decision,decision_notes
0,phase6_regime_context_alpha_20260504_185145,phase6_regime_overlay_diagnostic_v1,36,32,0,0,100,PARK_OVERLAYS,Regime overlays are diagnostic-only and should...


## 14. Final Diagnostic Decision


In [14]:
final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "regime_context_alpha_scoring_version", "value": REGIME_CONTEXT_ALPHA_SCORING_VERSION},
        {"metric": "regime_context_alpha_wfv_version", "value": REGIME_CONTEXT_ALPHA_WFV_VERSION},
        {"metric": "regime_overlay_diagnostic_version", "value": REGIME_OVERLAY_DIAGNOSTIC_VERSION},
        {"metric": "approved_regime_context_overlay_candidates", "value": len(approved_regime_context_names)},
        {"metric": "alpha_horizons_scored", "value": len(regime_context_alpha_scores)},
        {"metric": "wfv_candidate_rows", "value": len(regime_context_alpha_wfv_candidates)},
        {"metric": "wfv_windows", "value": len(regime_context_alpha_wfv_windows)},
        {"metric": "wfv_window_result_rows", "value": len(regime_context_alpha_wfv_window_results)},
        {"metric": "approved_wfv_winners", "value": len(regime_context_alpha_wfv_winner_summary)},
        {"metric": "diagnostic_decision", "value": diagnostic_decision},
    ]
)

full_wfv_gate_table = (
    regime_context_alpha_wfv_gate.sort_values("effective_mean_test_ic", ascending=False)
    if not regime_context_alpha_wfv_gate.empty
    else regime_context_alpha_wfv_gate
)

print("Regime overlay validation diagnostic summary")
display(final_summary)

print("Approved overlay count by overlay_type")
display(approved_overlay_count_by_overlay_type)

print("Approved overlay count by base_alpha_name")
display(approved_overlay_count_by_base_alpha_name)

print("V3 overlay inclusion check")
display(v3_overlay_inclusion_check)

print("WFV gate counts")
display(regime_context_alpha_wfv_gate_counts)

print("Winner summary")
display(regime_context_alpha_wfv_winner_summary)

print("Failure breakdown")
display(regime_context_alpha_wfv_failure_breakdown)

print("Diagnostic decision")
print(decision_notes)
display(regime_overlay_diagnostic_decision)

print("Approved regime-context overlay candidates")
display(approved_regime_context_quality.sort_values("alpha_name"))

print("Activation diagnostics")
display(activation_diagnostics.sort_values("alpha_name"))

print("Scoring gate counts")
display(scoring_gate_counts)

print("Full WFV gate table")
display(full_wfv_gate_table)

print("SQLite tables written")
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving regime context alpha validation outputs')


Regime overlay validation diagnostic summary


,metric,value
0,run_id,phase6_regime_context_alpha_20260504_185145
1,run_timestamp,2026-05-04 18:51:45
2,regime_context_alpha_scoring_version,phase6_regime_context_alpha_scoring_v2
3,regime_context_alpha_wfv_version,phase6_regime_context_alpha_wfv_v2
4,regime_overlay_diagnostic_version,phase6_regime_overlay_diagnostic_v1
5,approved_regime_context_overlay_candidates,32
6,alpha_horizons_scored,128
7,wfv_candidate_rows,100
8,wfv_windows,4
9,wfv_window_result_rows,400


Approved overlay count by overlay_type


,overlay_type,n_approved_overlays
0,base_passthrough,8
1,defensive_downscale,8
2,mild_regime_scaled,8
3,volatility_stress_scaled,8


Approved overlay count by base_alpha_name


...

,status,n_alpha_horizons
0,APPROVED_FOR_ALPHA_WFV,80
1,REJECTED_ALPHA_LOW_SIGNAL,28
2,WATCHLIST_ALPHA_WFV,20


Full WFV gate table


,alpha_name,horizon,candidate_tier,alpha_direction,alpha_strength,n_windows,mean_train_ic,mean_test_ic,effective_mean_train_ic,effective_mean_test_ic,median_test_ic,effective_median_test_ic,test_ic_std,effective_test_ic_std,test_ic_ir,effective_test_ic_ir,test_positive_ic_rate,persistence_ratio,sign_consistency,direction_flip_warning,degradation_ratio,n_positive_test_windows,n_negative_test_windows,abs_mean_test_ic,abs_test_ic_ir,status,wfv_gate_notes,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
99,alpha_smooth_regime_weighted_v2__volatility_st...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740579,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
96,alpha_smooth_regime_weighted_v2__mild_regime_s...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740554,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,mild_regime_scaled,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
90,alpha_smooth_regime_weighted_v2__base_passthrough,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180969,0.180969,0.128660,0.128660,1.145877,1.145877,0.75,0.50,0.75,True,6.740604,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,base_passthrough,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
93,alpha_smooth_regime_weighted_v2__defensive_dow...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021872,0.147428,0.021872,0.147428,0.180968,0.180968,0.128659,0.128659,1.145877,1.145877,0.75,0.50,0.75,True,6.740596,3,1,0.147428,1.145877,REJECTED_ALPHA_WFV,direction flip,defensive_downscale,alpha_smooth_regime_weighted_v2,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
35,alpha_health_weighted_research_v1__volatility_...,20,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,STRONG,4,0.021728,0.146375,0.021728,0.146375,0.178983,0.178983,0.127684,0.127684,1.146382,1.146382,0.75,0.50,0.75,True,6.736750,3,1,0.146375,1.146382,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_health_weighted_research_v1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,alpha_decay_aware_dynamic_v3__volatility_stres...,5,PRIMARY_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,MODERATE,4,0.013401,0.045324,0.013401,0.045324,0.057883,0.057883,0.063181,0.063181,0.717371,0.717371,0.75,0.75,0.75,True,3.382092,3,1,0.045324,0.717371,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
72,alpha_rolling_ic_dynamic_v3__base_passthrough,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,WEAK,4,0.006835,0.016490,0.006835,0.016490,0.026330,0.026330,0.029602,0.029602,0.557066,0.557066,0.75,1.00,0.75,True,2.412614,3,1,0.016490,0.557066,REJECTED_ALPHA_WFV,direction flip,base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
84,alpha_rolling_ic_dynamic_v3__volatility_stress...,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE,POSITIVE_EDGE,WEAK,4,0.006835,0.016490,0.006835,0.016490,0.026330,0.026330,0.029602,0.029602,0.557066,0.557066,0.75,1.00,0.75,True,2.412614,3,1,0.016490,0.557066,REJECTED_ALPHA_WFV,direction flip,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
80,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,1,WATCHLIST_REGIME_CONTEXT_ALPHA_WFV_CANDIDATE

SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,scores,regime_context_alpha_scores_current,regime_context_alpha_scores_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,scoring_gate,regime_context_alpha_scoring_gate_current,regime_context_alpha_scoring_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,best_horizon,regime_context_alpha_best_horizon_current,regime_context_alpha_best_horizon_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,wfv_windows,regime_context_alpha_wfv_windows_current,regime_context_alpha_wfv_windows_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,wfv_window_results,regime_context_alpha_wfv_window_results_current,regime_context_alpha_wfv_window_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,wfv_summary,regime_context_alpha_wfv_summary_current,regime_context_alpha_wfv_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,wfv_gate,regime_context_alpha_wfv_gate_current,regime_context_alpha_wfv_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
7,wfv_failure_breakdown,regime_context_alpha_wfv_failure_breakdown_cur...,regime_context_alpha_wfv_failure_breakdown_his...,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
8,wfv_winner_summary,regime_context_alpha_wfv_winner_summary_current,regime_context_alpha_wfv_winner_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
9,diagnostic_decision,regime_overlay_diagnostic_decision_current,regime_overlay_diagnostic_decision_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [15]:
from src.db import load_table

score_gate = load_table("regime_context_alpha_scoring_gate_current")
wfv_gate = load_table("regime_context_alpha_wfv_gate_current")
winners = load_table("regime_context_alpha_wfv_winner_summary_current")
failure = load_table("regime_context_alpha_wfv_failure_breakdown_current")
diagnostic_decision_readback = load_table("regime_overlay_diagnostic_decision_current")

print("Scoring gate status counts")
display(score_gate["status"].value_counts())

print("WFV gate status counts")
display(wfv_gate["status"].value_counts())

print("Current diagnostic decision")
display(diagnostic_decision_readback)

print("Current WFV gate by overlay")
display(
    wfv_gate.sort_values("effective_mean_test_ic", ascending=False)[
        [
            "alpha_name",
            "overlay_type",
            "base_alpha_name",
            "horizon",
            "effective_mean_test_ic",
            "effective_test_ic_ir",
            "persistence_ratio",
            "sign_consistency",
            "status",
            "wfv_gate_notes",
        ]
    ]
)

print("Winner summary")
display(winners)

print("Failure breakdown")
display(failure)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after regime overlay validation diagnostics readback')


Scoring gate status counts


status
APPROVED_FOR_ALPHA_WFV       80
REJECTED_ALPHA_LOW_SIGNAL    28
WATCHLIST_ALPHA_WFV          20
Name: count, dtype: int64

WFV gate status counts


status
REJECTED_ALPHA_WFV    100
Name: count, dtype: int64

Current diagnostic decision


...

,alpha_name,overlay_type,base_alpha_name,horizon,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,status,wfv_gate_notes
99,alpha_smooth_regime_weighted_v2__volatility_st...,volatility_stress_scaled,alpha_smooth_regime_weighted_v2,20,0.147428,1.145877,0.50,0.75,REJECTED_ALPHA_WFV,direction flip
96,alpha_smooth_regime_weighted_v2__mild_regime_s...,mild_regime_scaled,alpha_smooth_regime_weighted_v2,20,0.147428,1.145877,0.50,0.75,REJECTED_ALPHA_WFV,direction flip
90,alpha_smooth_regime_weighted_v2__base_passthrough,base_passthrough,alpha_smooth_regime_weighted_v2,20,0.147428,1.145877,0.50,0.75,REJECTED_ALPHA_WFV,direction flip
93,alpha_smooth_regime_weighted_v2__defensive_dow...,defensive_downscale,alpha_smooth_regime_weighted_v2,20,0.147428,1.145877,0.50,0.75,REJECTED_ALPHA_WFV,direction flip
35,alpha_health_weighted_research_v1__volatility_...,volatility_stress_scaled,alpha_health_weighted_research_v1,20,0.146375,1.146382,0.50,0.75,REJECTED_ALPHA_WFV,direction flip
...,...,...,...,...,...,...,...,...,...,...
9,alpha_decay_aware_dynamic_v3__volatility_stres...,volatility_stress_scaled,alpha_decay_aware_dynamic_v3,5,0.045324,0.717371,0.75,0.75,REJECTED_ALPHA_WFV,direction flip
72,alpha_rolling_ic_dynamic_v3__base_passthrough,base_passthrough,alpha_rolling_ic_dynamic_v3,1,0.016490,0.557066,1.00,0.75,REJECTED_ALPHA_WFV,direction flip
84,alpha_rolling_ic_dynamic_v3__volatility_stress...,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,1,0.016490,0.557066,1.00,0.75,REJECTED_ALPHA_WFV,direction flip
80,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,1,0.016490,0.557066,1.00,0.75,REJECTED_ALPHA_WFV,direction flip


Winner summary


,alpha_name,horizon,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,candidate_tier,alpha_direction,alpha_strength,status,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio,run_id,regime_context_alpha_scoring_version,regime_context_alpha_wfv_version


Failure breakdown


,failure_reason,count,pct_of_candidates,run_id,regime_context_alpha_scoring_version,regime_context_alpha_wfv_version
0,direction flip,100,1.0,phase6_regime_context_alpha_20260504_185145,phase6_regime_context_alpha_scoring_v2,phase6_regime_context_alpha_wfv_v2
